# XAUUSD — New York Session Range Breakout Strategy
## بک‌تست استراتژی بریک‌اوت محدوده سشن نیویورک — تایم‌فریم M1

---
### استراتژی
1. ساعت **19:00** در هر روز = شروع سشن نیویورک (broker time as stored in CSV — no conversion)
2. **5 کندل اول M1** (19:00–19:04) → **Zone High** و **Zone Low**
3. بعد از بسته شدن کندل پنجم (19:05) منتظر بریک‌اوت M1:
   - **Scenario 1 — Fake Break**: شکست زون → برگشت داخل → شکست دوم → ورود
   - **Scenario 2 — Strong Breakout**: شکست + کندل تأیید کامل خارج زون → ورود
4. **SL**: نزدیک‌ترین Swing قبل از ورود (M1)
5. **TP**: Entry ± 2R

---
| پارامتر | مقدار |
|---|---|
| نماد | XAUUSD |
| تایم‌فریم | M1 |
| شروع سشن NY | 19:00 (CSV broker time) |
| کندل‌های Zone | 5 اول M1 (19:00–19:04) |
| Risk/Reward | 1:2 |


## Step 1 — Install & Imports


In [221]:
%pip install plotly


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [222]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
from pathlib import Path
from typing import Optional

import plotly
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

pio.renderers.default = "notebook"
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", "{:.4f}".format)

print("Libraries loaded")
print(f"  Pandas : {pd.__version__}")
print(f"  NumPy  : {np.__version__}")
print(f"  Plotly : {plotly.__version__}")


Libraries loaded
  Pandas : 3.0.2
  NumPy  : 2.2.6
  Plotly : 6.7.0


## Step 2 — Configuration


In [223]:
# ── Data ─────────────────────────────────────────────────────────────────────
SYMBOL        = "BTCUSD"
DATA_DIR      = Path("./data")
LOOKBACK_DAYS = 30

# ── Session (CSV broker time — 19:00 = NY open, no tz conversion) ────────────
NY_OPEN_HOUR  = 19
NY_OPEN_MIN   = 0
NY_OPEN_TOD   = NY_OPEN_HOUR * 60 + NY_OPEN_MIN   # 1140 min since midnight

# ── Zone ─────────────────────────────────────────────────────────────────────
ZONE_CANDLES  = 5    # first 5 M1 candles define zone (19:00–19:04)

# ── Swing Detection ──────────────────────────────────────────────────────────
SWING_WINDOW     = 5
SWING_LOOKBACK_H = 4

# ── Risk Management ──────────────────────────────────────────────────────────
RISK_REWARD = 2.5
SL_BUFFER   = 0.30

# ── Simulation ───────────────────────────────────────────────────────────────
MAX_TRADE_HOURS = 48

print("Configuration:")
print(f"  Symbol        : {SYMBOL}")
print(f"  Lookback      : {LOOKBACK_DAYS} days")
print(f"  NY Open (CSV) : {NY_OPEN_HOUR:02d}:{NY_OPEN_MIN:02d}")
print(f"  Zone candles  : first {ZONE_CANDLES} M1 candles (19:00–19:04)")
print(f"  Swing window  : {SWING_WINDOW} bars each side")
print(f"  SL buffer     : {SL_BUFFER}")
print(f"  Risk/Reward   : 1:{RISK_REWARD}")


Configuration:
  Symbol        : BTCUSD
  Lookback      : 30 days
  NY Open (CSV) : 19:00
  Zone candles  : first 5 M1 candles (19:00–19:04)
  Swing window  : 5 bars each side
  SL buffer     : 0.3
  Risk/Reward   : 1:2.5


## Step 3 — Load M1 Data

داده‌های M1 از فایل CSV خوانده می‌شوند.
**زمان‌ها بدون هیچ تغییری** استفاده می‌شوند — 19:00 در CSV = شروع سشن نیویورک.


In [224]:
def load_ohlcv(symbol: str, tf: str) -> pd.DataFrame:
    """Load CSV — timestamps used as-is (no timezone shift)."""
    path = DATA_DIR / symbol / tf / "ohlcv.csv"
    if not path.exists():
        raise FileNotFoundError(f"Missing: {path}")
    df = pd.read_csv(path)
    df["time"] = pd.to_datetime(df["time"], utc=True)
    df = df.sort_values("time").reset_index(drop=True)
    keep = ["time", "open", "high", "low", "close", "tick_volume"]
    df = df[[c for c in keep if c in df.columns]].copy()
    df.rename(columns={"tick_volume": "volume"}, inplace=True)
    return df


df_m1_raw = load_ohlcv(SYMBOL, "M1")

cutoff = pd.Timestamp.now(tz="UTC") - pd.Timedelta(days=LOOKBACK_DAYS)
df_m1 = df_m1_raw[df_m1_raw["time"] >= cutoff].copy().reset_index(drop=True)

print(f"M1 raw           : {len(df_m1_raw):>10,} bars")
print(f"M1 (lookback {LOOKBACK_DAYS}d) : {len(df_m1):>7,} bars")
print(f"  From : {df_m1['time'].min()}")
print(f"  To   : {df_m1['time'].max()}")
df_m1.tail(3)


M1 raw           :  3,817,983 bars
M1 (lookback 30d) :  42,136 bars
  From : 2026-04-16 14:57:00+00:00
  To   : 2026-05-16 17:51:00+00:00


,time,open,high,low,close,volume
42133,2026-05-16 17:49:00+00:00,78171.7900,78173.1100,78153.3100,78161.3700,251
42134,2026-05-16 17:50:00+00:00,78161.7600,78162.6100,78136.4200,78136.6000,242
42135,2026-05-16 17:51:00+00:00,78136.5400,78161.2100,78135.6500,78160.8900,202


## Step 4 — Time Columns & Session Detection


In [225]:
def add_time_cols(df: pd.DataFrame) -> pd.DataFrame:
    """Add date/hour/minute/weekday from raw CSV timestamps (no tz conversion)."""
    df = df.copy()
    df["date"]    = df["time"].dt.date
    df["hour"]    = df["time"].dt.hour
    df["minute"]  = df["time"].dt.minute
    df["weekday"] = df["time"].dt.weekday   # 0=Mon, 4=Fri
    df["tod_min"] = df["hour"] * 60 + df["minute"]
    return df


def mark_session(df: pd.DataFrame) -> pd.DataFrame:
    """Flag candles at or after 19:00 on weekdays."""
    df = df.copy()
    in_window  = df["tod_min"] >= NY_OPEN_TOD
    is_weekday = df["weekday"] < 5
    df["in_session"] = in_window & is_weekday
    return df


df_m1 = add_time_cols(df_m1)
df_m1 = mark_session(df_m1)

trading_days = sorted(df_m1[df_m1["in_session"]]["date"].unique())
print(f"Trading days with session data: {len(trading_days)}")
print(f"First 5 : {trading_days[:5]}")
print(f"Last 5  : {trading_days[-5:]}")

sample = df_m1[(df_m1["hour"] == NY_OPEN_HOUR) & (df_m1["minute"] == NY_OPEN_MIN)].head(3)
print("\nSample M1 candles at 19:00 (NY session open):")
print(sample[["time", "open", "high", "low", "close"]].to_string())


Trading days with session data: 22
First 5 : [datetime.date(2026, 4, 16), datetime.date(2026, 4, 17), datetime.date(2026, 4, 20), datetime.date(2026, 4, 21), datetime.date(2026, 4, 22)]
Last 5  : [datetime.date(2026, 5, 11), datetime.date(2026, 5, 12), datetime.date(2026, 5, 13), datetime.date(2026, 5, 14), datetime.date(2026, 5, 15)]

Sample M1 candles at 19:00 (NY session open):
                          time       open       high        low      close
243  2026-04-16 19:00:00+00:00 74683.6800 74748.7900 74683.1000 74740.8600
1683 2026-04-17 19:00:00+00:00 77771.5700 77772.1400 77691.2500 77711.9200
3112 2026-04-18 19:00:00+00:00 76153.5200 76154.5400 76018.9300 76019.1700


## Step 5 — Build Zone (First 5 M1 Candles)

برای هر روز معاملاتی:
1. ۵ کندل اول M1 بعد از ۱۹:۰۰ (19:00، 19:01، 19:02، 19:03، 19:04)
2. **Zone High** = بالاترین High این ۵ کندل
3. **Zone Low** = پایین‌ترین Low این ۵ کندل
4. زون بعد از بسته شدن کندل پنجم **(19:05)** فعال می‌شود


In [226]:
def build_zones(df_m1: pd.DataFrame) -> pd.DataFrame:
    """Extract Zone High/Low from first 5 M1 candles of each NY session."""
    records = []

    for day in trading_days:
        # Candles at minutes 0,1,2,3,4 of hour 19 = 19:00–19:04
        zone_candles = df_m1[
            (df_m1["date"] == day) &
            (df_m1["hour"] == NY_OPEN_HOUR) &
            (df_m1["minute"] >= NY_OPEN_MIN) &
            (df_m1["minute"] < NY_OPEN_MIN + ZONE_CANDLES)
        ].copy()

        if zone_candles.empty:
            continue

        zone_high = zone_candles["high"].max()
        zone_low  = zone_candles["low"].min()
        last_time        = zone_candles["time"].max()
        zone_active_from = last_time + pd.Timedelta(minutes=1)  # 19:05

        records.append({
            "date"             : day,
            "zone_high"        : round(zone_high, 2),
            "zone_low"         : round(zone_low,  2),
            "zone_mid"         : round((zone_high + zone_low) / 2, 2),
            "zone_width"       : round(zone_high - zone_low, 2),
            "zone_open_time"   : zone_candles["time"].min(),
            "zone_active_from" : zone_active_from,
            "zone_candle_count": len(zone_candles),
            "zone_bullish"     : bool(
                zone_candles.iloc[-1]["close"] >= zone_candles.iloc[0]["open"]
            ),
        })

    zones = pd.DataFrame(records)
    print(f"Zones extracted : {len(zones)}")
    print(f"Full 5-candle   : {(zones['zone_candle_count']==ZONE_CANDLES).sum()}")
    print(f"Width  avg={zones['zone_width'].mean():.2f}  "
          f"min={zones['zone_width'].min():.2f}  "
          f"max={zones['zone_width'].max():.2f}")
    print(f"Bullish zones   : {zones['zone_bullish'].sum()} "
          f"({zones['zone_bullish'].mean()*100:.0f}%)")
    return zones


zones_df = build_zones(df_m1)
zones_df.head(5)


Zones extracted : 22
Full 5-candle   : 22
Width  avg=156.37  min=70.81  max=232.86
Bullish zones   : 13 (59%)


,date,zone_high,zone_low,zone_mid,zone_width,zone_open_time,zone_active_from,zone_candle_count,zone_bullish
0,2026-04-16,74801.4900,74683.1000,74742.3000,118.3900,2026-04-16 19:00:00+00:00,2026-04-16 19:05:00+00:00,5,True
1,2026-04-17,77908.1300,77691.2500,77799.6900,216.8800,2026-04-17 19:00:00+00:00,2026-04-17 19:05:00+00:00,5,True
2,2026-04-20,75710.6400,75503.4400,75607.0400,207.2000,2026-04-20 19:00:00+00:00,2026-04-20 19:05:00+00:00,5,False
3,2026-04-21,75987.9600,75818.2400,75903.1000,169.7200,2026-04-21 19:00:00+00:00,2026-04-21 19:05:00+00:00,5,True
4,2026-04-22,79422.9300,79259.8300,79341.3800,163.1000,2026-04-22 19:00:00+00:00,2026-04-22 19:05:00+00:00,5,True


### Zone Visualization — M1

۵ کندل Zone (آبی) و خطوط High/Low آن روی چارت **M1** نشان داده می‌شوند.


In [227]:
def plot_zone_day(day, df_m1: pd.DataFrame, zones: pd.DataFrame,
                  show_bars: int = 60) -> None:
    """Plot M1 chart with first-5-candle zone highlighted."""
    zone = zones[zones["date"] == day]
    if zone.empty:
        print(f"No zone for {day}")
        return
    zone = zone.iloc[0]

    day_data = df_m1[
        (df_m1["date"] == day) &
        (df_m1["time"] >= zone["zone_open_time"])
    ].head(show_bars).copy()

    if day_data.empty:
        print(f"No M1 data for {day}")
        return

    fig = go.Figure()
    fig.add_trace(go.Candlestick(
        x=day_data["time"], open=day_data["open"],
        high=day_data["high"], low=day_data["low"], close=day_data["close"],
        name="XAUUSD M1",
        increasing_line_color="#26a69a", decreasing_line_color="#ef5350",
    ))

    # Highlight zone candles (19:00–19:04)
    fig.add_vrect(
        x0=zone["zone_open_time"], x1=zone["zone_active_from"],
        fillcolor="rgba(33,150,243,0.20)", line_width=0,
        annotation_text=f"Zone ({ZONE_CANDLES} M1 candles)",
        annotation_position="top left",
    )
    fig.add_hline(y=zone["zone_high"], line_color="#2196F3", line_dash="dash", line_width=2,
                  annotation_text=f"High {zone['zone_high']:.2f}",
                  annotation_position="top right")
    fig.add_hline(y=zone["zone_low"], line_color="#FF9800", line_dash="dash", line_width=2,
                  annotation_text=f"Low {zone['zone_low']:.2f}",
                  annotation_position="bottom right")
    fig.add_hrect(y0=zone["zone_low"], y1=zone["zone_high"],
                  fillcolor="rgba(33,150,243,0.07)", line_width=0)

    fig.update_layout(
        title=f"XAUUSD M1 — NY Zone (first {ZONE_CANDLES} candles at 19:00) — {day}",
        xaxis_rangeslider_visible=False,
        template="plotly_dark", height=450,
        yaxis_title="Price (USD)",
    )
    fig.show()


# Show last 5 trading days
for day in zones_df["date"].tail(5):
    plot_zone_day(day, df_m1, zones_df)


## Step 6 — Breakout Detection Helpers (M1)

بریک‌اوت با **Close کندل M1 خارج** از Zone:
- **Break UP**: `close > zone_high`
- **Break DOWN**: `close < zone_low`


In [228]:
def is_strong_confirm_up(candle, zh: float) -> bool:
    """Scenario-2 BUY confirmation: both close AND low above zone_high."""
    return candle["close"] > zh and candle["low"] > zh


def is_strong_confirm_dn(candle, zl: float) -> bool:
    """Scenario-2 SELL confirmation: both close AND high below zone_low."""
    return candle["close"] < zl and candle["high"] < zl


def returned_to_zone_up(candle, zh: float) -> bool:
    """After up-break, low touched or re-entered zone."""
    return candle["low"] <= zh


def returned_to_zone_dn(candle, zl: float) -> bool:
    """After down-break, high touched or re-entered zone."""
    return candle["high"] >= zl


print("Breakout helpers defined:")
print("  is_strong_confirm_up  → Scenario-2 BUY  (low > zone_high)")
print("  is_strong_confirm_dn  → Scenario-2 SELL (high < zone_low)")
print("  returned_to_zone_up   → fake-break detection after up-break")
print("  returned_to_zone_dn   → fake-break detection after down-break")


Breakout helpers defined:
  is_strong_confirm_up  → Scenario-2 BUY  (low > zone_high)
  is_strong_confirm_dn  → Scenario-2 SELL (high < zone_low)
  returned_to_zone_up   → fake-break detection after up-break
  returned_to_zone_dn   → fake-break detection after down-break


## Step 7 — Swing Detection (M1)

برای محاسبه SL، نزدیک‌ترین Swing قبل از ورود روی M1.


In [229]:
def _swing_lows(lows: np.ndarray, window: int) -> np.ndarray:
    n, out = len(lows), np.full(len(lows), np.nan)
    for i in range(window, n - window):
        nb = np.concatenate([lows[i-window:i], lows[i+1:i+window+1]])
        if lows[i] <= nb.min():
            out[i] = lows[i]
    return out


def _swing_highs(highs: np.ndarray, window: int) -> np.ndarray:
    n, out = len(highs), np.full(len(highs), np.nan)
    for i in range(window, n - window):
        nb = np.concatenate([highs[i-window:i], highs[i+1:i+window+1]])
        if highs[i] >= nb.max():
            out[i] = highs[i]
    return out


def nearest_swing_low(df_lb: pd.DataFrame, window: int = SWING_WINDOW) -> float:
    if len(df_lb) < window * 2 + 1:
        return float(df_lb["low"].min()) if len(df_lb) else np.nan
    valid = [v for v in _swing_lows(df_lb["low"].values, window) if not np.isnan(v)]
    return valid[-1] if valid else float(df_lb["low"].min())


def nearest_swing_high(df_lb: pd.DataFrame, window: int = SWING_WINDOW) -> float:
    if len(df_lb) < window * 2 + 1:
        return float(df_lb["high"].max()) if len(df_lb) else np.nan
    valid = [v for v in _swing_highs(df_lb["high"].values, window) if not np.isnan(v)]
    return valid[-1] if valid else float(df_lb["high"].max())


def get_lookback(df_m1: pd.DataFrame, entry_time: pd.Timestamp,
                 hours: float = SWING_LOOKBACK_H) -> pd.DataFrame:
    start = entry_time - pd.Timedelta(hours=hours)
    return df_m1[(df_m1["time"] >= start) & (df_m1["time"] < entry_time)]


print("Swing detection functions defined.")


Swing detection functions defined.


### Swing Visualization — M1


In [230]:
def plot_swings_demo(day, df_m1: pd.DataFrame, zones: pd.DataFrame,
                     window: int = SWING_WINDOW, show_bars: int = 120) -> None:
    zone = zones[zones["date"] == day]
    if zone.empty:
        return
    zone = zone.iloc[0]

    day_m1 = df_m1[
        (df_m1["date"] == day) & df_m1["in_session"]
    ].head(show_bars).reset_index(drop=True)

    if len(day_m1) < window * 2 + 2:
        print(f"Not enough M1 bars for {day}")
        return

    sl_arr  = _swing_lows(day_m1["low"].values, window)
    sh_arr  = _swing_highs(day_m1["high"].values, window)
    sl_mask = ~np.isnan(sl_arr)
    sh_mask = ~np.isnan(sh_arr)

    fig = go.Figure()
    fig.add_trace(go.Candlestick(
        x=day_m1["time"], open=day_m1["open"], high=day_m1["high"],
        low=day_m1["low"], close=day_m1["close"], name="M1",
        increasing_line_color="#26a69a", decreasing_line_color="#ef5350",
    ))
    if sl_mask.any():
        fig.add_trace(go.Scatter(
            x=day_m1["time"][sl_mask], y=sl_arr[sl_mask],
            mode="markers", name="Swing Low",
            marker=dict(symbol="triangle-up", size=12, color="#00E676",
                        line=dict(color="white", width=1)),
        ))
    if sh_mask.any():
        fig.add_trace(go.Scatter(
            x=day_m1["time"][sh_mask], y=sh_arr[sh_mask],
            mode="markers", name="Swing High",
            marker=dict(symbol="triangle-down", size=12, color="#FF1744",
                        line=dict(color="white", width=1)),
        ))
    fig.add_hline(y=zone["zone_high"], line_color="#2196F3", line_dash="dash", line_width=1.5)
    fig.add_hline(y=zone["zone_low"],  line_color="#FF9800", line_dash="dash", line_width=1.5)
    fig.add_hrect(y0=zone["zone_low"], y1=zone["zone_high"],
                  fillcolor="rgba(33,150,243,0.07)", line_width=0)
    fig.update_layout(
        title=f"Swing Detection (window={window}) — M1 — {day}",
        xaxis_rangeslider_visible=False,
        template="plotly_dark", height=480,
    )
    fig.show()


plot_swings_demo(zones_df["date"].iloc[0], df_m1, zones_df)


## Step 8 — Risk Management

| پارامتر | فرمول |
|---|---|
| **SL (BUY)**  | `min(swing_low, zone_low) - buffer` |
| **SL (SELL)** | `max(swing_high, zone_high) + buffer` |
| **Risk (R)**  | `abs(Entry - SL)` |
| **TP (BUY)**  | `Entry + 2 × Risk` |
| **TP (SELL)** | `Entry - 2 × Risk` |


In [231]:
def calc_sl_tp(direction: str, entry: float, lookback_m1: pd.DataFrame,
               zone_high: float, zone_low: float) -> tuple:
    if direction == "BUY":
        sl_swing = nearest_swing_low(lookback_m1)
        sl   = min(sl_swing, zone_low) - SL_BUFFER
        risk = entry - sl
        if risk <= 0:
            return None, None, None
        tp = entry + RISK_REWARD * risk
    else:
        sl_swing = nearest_swing_high(lookback_m1)
        sl   = max(sl_swing, zone_high) + SL_BUFFER
        risk = sl - entry
        if risk <= 0:
            return None, None, None
        tp = entry - RISK_REWARD * risk
    return round(sl, 2), round(tp, 2), round(risk, 2)


print("calc_sl_tp() defined.")
print("  BUY  : SL = min(swing_low, zone_low) - buffer  |  TP = entry + 2*risk")
print("  SELL : SL = max(swing_high, zone_high) + buffer |  TP = entry - 2*risk")


calc_sl_tp() defined.
  BUY  : SL = min(swing_low, zone_low) - buffer  |  TP = entry + 2*risk
  SELL : SL = max(swing_high, zone_high) + buffer |  TP = entry - 2*risk


## Step 9 — Backtesting Engine (M1 State Machine)

```
IDLE  (starts at 19:05, after zone formed)
  ↓ close > zone_high              ↓ close < zone_low
FIRST_BREAK_UP               FIRST_BREAK_DOWN
  ↓ low <= zone_high               ↓ high >= zone_low
RETURNED_AFTER_UP            RETURNED_AFTER_DOWN
  ↓ close > zone_high              ↓ close < zone_low
[ENTRY BUY — Scenario 1]     [ENTRY SELL — Scenario 1]

FIRST_BREAK_UP  + low > zone_high  → [ENTRY BUY  — Scenario 2]
FIRST_BREAK_DOWN + high < zone_low → [ENTRY SELL — Scenario 2]
```

**Anti-lookahead**: کندل‌ها به‌صورت sequential پردازش، فقط Close قطعی استفاده می‌شود.


In [232]:
def simulate_trade(df_m1_full: pd.DataFrame, entry_time: pd.Timestamp,
                   direction: str, entry: float, sl: float, tp: float) -> dict:
    """Walk M1 bars forward; pessimistic fill (SL wins if both hit same bar)."""
    end_time = entry_time + pd.Timedelta(hours=MAX_TRADE_HOURS)
    bars = df_m1_full[
        (df_m1_full["time"] > entry_time) &
        (df_m1_full["time"] <= end_time)
    ]
    for _, bar in bars.iterrows():
        held = int((bar["time"] - entry_time).total_seconds() / 60)
        if direction == "BUY":
            if bar["low"] <= sl:
                return {"result":"SL","exit_price":sl,"exit_time":bar["time"],"pnl_r":-1.0,"bars_held":held}
            if bar["high"] >= tp:
                return {"result":"TP","exit_price":tp,"exit_time":bar["time"],"pnl_r":RISK_REWARD,"bars_held":held}
        else:
            if bar["high"] >= sl:
                return {"result":"SL","exit_price":sl,"exit_time":bar["time"],"pnl_r":-1.0,"bars_held":held}
            if bar["low"] <= tp:
                return {"result":"TP","exit_price":tp,"exit_time":bar["time"],"pnl_r":RISK_REWARD,"bars_held":held}
    if not bars.empty:
        last = bars.iloc[-1]
        risk = abs(entry - sl)
        pnl  = (last["close"]-entry)/risk if direction=="BUY" else (entry-last["close"])/risk
        return {"result":"OPEN","exit_price":round(last["close"],2),
                "exit_time":last["time"],"pnl_r":round(pnl,3),"bars_held":len(bars)}
    return {"result":"OPEN","exit_price":entry,"exit_time":entry_time,"pnl_r":0.0,"bars_held":0}


print("simulate_trade() defined.")


simulate_trade() defined.


In [233]:
def _record(zone, direction, scenario, entry_candle, sl, tp, risk, outcome) -> dict:
    return {
        "date"        : zone["date"],
        "direction"   : direction,
        "scenario"    : scenario,
        "entry_time"  : entry_candle["time"],
        "entry_price" : round(entry_candle["close"], 2),
        "sl"          : sl,
        "tp"          : tp,
        "risk"        : risk,
        "zone_high"   : zone["zone_high"],
        "zone_low"    : zone["zone_low"],
        "zone_width"  : zone["zone_width"],
        **outcome,
    }


def run_backtest(df_m1: pd.DataFrame, zones_df: pd.DataFrame) -> pd.DataFrame:
    """M1-only backtesting engine. One trade per zone per day."""
    trades = []

    for _, zone in zones_df.iterrows():
        day              = zone["date"]
        zh               = zone["zone_high"]
        zl               = zone["zone_low"]
        zone_active_from = zone["zone_active_from"]

        # M1 candles for state machine: session candles after zone is formed
        day_m1 = df_m1[
            (df_m1["date"] == day) &
            df_m1["in_session"] &
            (df_m1["time"] >= zone_active_from)
        ].reset_index(drop=True)

        if len(day_m1) < 3:
            continue

        state = "IDLE"

        for i in range(len(day_m1)):
            c = day_m1.iloc[i]

            if state == "IDLE":
                if c["close"] > zh:
                    state = "FIRST_BREAK_UP"
                elif c["close"] < zl:
                    state = "FIRST_BREAK_DOWN"

            elif state == "FIRST_BREAK_UP":
                if returned_to_zone_up(c, zh):
                    state = "RETURNED_AFTER_UP"
                elif is_strong_confirm_up(c, zh):
                    lb = get_lookback(df_m1, c["time"])
                    sl, tp, risk = calc_sl_tp("BUY", c["close"], lb, zh, zl)
                    if sl is None: continue
                    outcome = simulate_trade(df_m1, c["time"], "BUY", c["close"], sl, tp)
                    trades.append(_record(zone, "BUY", "STRONG_BREAKOUT", c, sl, tp, risk, outcome))
                    break

            elif state == "FIRST_BREAK_DOWN":
                if returned_to_zone_dn(c, zl):
                    state = "RETURNED_AFTER_DOWN"
                elif is_strong_confirm_dn(c, zl):
                    lb = get_lookback(df_m1, c["time"])
                    sl, tp, risk = calc_sl_tp("SELL", c["close"], lb, zh, zl)
                    if sl is None: continue
                    outcome = simulate_trade(df_m1, c["time"], "SELL", c["close"], sl, tp)
                    trades.append(_record(zone, "SELL", "STRONG_BREAKOUT", c, sl, tp, risk, outcome))
                    break

            elif state == "RETURNED_AFTER_UP":
                if c["close"] > zh:
                    lb = get_lookback(df_m1, c["time"])
                    sl, tp, risk = calc_sl_tp("BUY", c["close"], lb, zh, zl)
                    if sl is None: continue
                    outcome = simulate_trade(df_m1, c["time"], "BUY", c["close"], sl, tp)
                    trades.append(_record(zone, "BUY", "FAKE_BREAK", c, sl, tp, risk, outcome))
                    break
                elif c["close"] < zl:
                    state = "FIRST_BREAK_DOWN"

            elif state == "RETURNED_AFTER_DOWN":
                if c["close"] < zl:
                    lb = get_lookback(df_m1, c["time"])
                    sl, tp, risk = calc_sl_tp("SELL", c["close"], lb, zh, zl)
                    if sl is None: continue
                    outcome = simulate_trade(df_m1, c["time"], "SELL", c["close"], sl, tp)
                    trades.append(_record(zone, "SELL", "FAKE_BREAK", c, sl, tp, risk, outcome))
                    break
                elif c["close"] > zh:
                    state = "FIRST_BREAK_UP"

    return pd.DataFrame(trades) if trades else pd.DataFrame()


print("Backtesting engine defined.")


Backtesting engine defined.


In [234]:
%%time
trades_df = run_backtest(df_m1, zones_df)

if trades_df.empty:
    print("No trades generated — check zone detection and data range.")
else:
    print(f"Backtest complete: {len(trades_df)} trades")
    print(f"  BUY  : {(trades_df['direction']=='BUY').sum()}")
    print(f"  SELL : {(trades_df['direction']=='SELL').sum()}")
    print()
    print("Results:")
    print(f"  TP   : {(trades_df['result']=='TP').sum()}")
    print(f"  SL   : {(trades_df['result']=='SL').sum()}")
    print(f"  OPEN : {(trades_df['result']=='OPEN').sum()}")
    print()
    print("By Scenario:")
    print(trades_df.groupby(["scenario","result"]).size().unstack(fill_value=0))

trades_df.head(5)


Backtest complete: 22 trades
  BUY  : 9
  SELL : 13

Results:
  TP   : 5
  SL   : 17
  OPEN : 0

By Scenario:
result           SL  TP
scenario               
FAKE_BREAK        8   2
STRONG_BREAKOUT   9   3
CPU times: total: 891 ms
Wall time: 1.21 s


,date,direction,scenario,entry_time,entry_price,sl,tp,risk,zone_high,zone_low,zone_width,result,exit_price,exit_time,pnl_r,bars_held
0,2026-04-16,BUY,STRONG_BREAKOUT,2026-04-16 19:08:00+00:00,74871.0500,74443.6000,75939.6800,427.4500,74801.4900,74683.1000,118.3900,SL,74443.6000,2026-04-16 19:33:00+00:00,-1.0000,25
1,2026-04-17,BUY,STRONG_BREAKOUT,2026-04-17 19:07:00+00:00,78002.2000,77690.9500,78780.3200,311.2500,77908.1300,77691.2500,216.8800,SL,77690.9500,2026-04-17 20:08:00+00:00,-1.0000,61
2,2026-04-20,SELL,STRONG_BREAKOUT,2026-04-20 19:09:00+00:00,75448.9500,75717.1700,74778.4000,268.2200,75710.6400,75503.4400,207.2000,SL,75717.1700,2026-04-20 20:18:00+00:00,-1.0000,69
3,2026-04-21,BUY,STRONG_BREAKOUT,2026-04-21 19:35:00+00:00,76128.1200,75817.9400,76903.5700,310.1800,75987.9600,75818.2400,169.7200,SL,75817.9400,2026-04-21 19:58:00+00:00,-1.0000,23
4,2026-04-22,SELL,STRONG_BREAKOUT,2026-04-22 19:11:00+00:00,79178.5700,79459.6400,78475.9000,281.0700,79422.9300,79259.8300,163.1000,TP,78475.9000,2026-04-22 23:22:00+00:00,2.5000,251


## Step 10 — Performance Metrics


In [235]:
def performance_metrics(trades_df: pd.DataFrame) -> dict:
    if trades_df.empty:
        return {}
    closed = trades_df[trades_df["result"].isin(["TP","SL"])].copy()
    open_count = (trades_df["result"]=="OPEN").sum()
    if closed.empty:
        print(f"No closed trades yet ({open_count} still OPEN — SL/TP not hit within {MAX_TRADE_HOURS}h window).")
        return {
            "total_trades": 0, "open_trades": open_count,
            "wins": 0, "losses": 0,
            "win_rate": 0.0, "total_r": 0.0, "avg_r": 0.0,
            "profit_factor": 0.0, "max_drawdown_r": 0.0,
            "max_consec_losses": 0, "avg_bars_held": 0.0,
            "scenario_stats": {},
        }
    n      = len(closed)
    wins   = (closed["result"]=="TP").sum()
    losses = (closed["result"]=="SL").sum()
    cum_r  = closed["pnl_r"].cumsum().reset_index(drop=True)
    dd     = cum_r - cum_r.cummax()
    pos_r  = closed[closed["pnl_r"]>0]["pnl_r"].sum()
    neg_r  = abs(closed[closed["pnl_r"]<0]["pnl_r"].sum())
    arr = (closed["result"]=="SL").astype(int).values
    max_cl = streak = 0
    for v in arr:
        streak = streak+1 if v==1 else 0
        max_cl = max(max_cl, streak)
    sc_stats = {}
    for sc in closed["scenario"].unique():
        sub = closed[closed["scenario"]==sc]
        sc_stats[sc] = {"n":len(sub),"wins":(sub["result"]=="TP").sum(),
                        "win_rate":(sub["result"]=="TP").mean(),"avg_r":sub["pnl_r"].mean()}
    return {
        "total_trades"      : n,
        "open_trades"       : open_count,
        "wins"              : int(wins),
        "losses"            : int(losses),
        "win_rate"          : wins/n,
        "total_r"           : round(closed["pnl_r"].sum(), 3),
        "avg_r"             : round(closed["pnl_r"].mean(), 3),
        "profit_factor"     : round(pos_r/neg_r, 3) if neg_r>0 else float("inf"),
        "max_drawdown_r"    : round(dd.min(), 3),
        "max_consec_losses" : max_cl,
        "avg_bars_held"     : round(closed["bars_held"].mean(), 1),
        "cum_r"             : cum_r,
        "drawdown"          : dd,
        "closed"            : closed,
        "scenario_stats"    : sc_stats,
    }


metrics = performance_metrics(trades_df)

if metrics:
    sep = "=" * 50
    print(sep)
    print("  PERFORMANCE — NY Session M1 Breakout")
    print(sep)
    print(f"  Total Trades         : {metrics['total_trades']}")
    print(f"  Open (not closed)    : {metrics['open_trades']}")
    if metrics["total_trades"] == 0:
        print("  No closed trades — all positions still OPEN or no entries found.")
        print("  Tip: increase MAX_TRADE_HOURS or check if 19:00 data exists for this symbol.")
    else:
        print(f"  Wins / Losses        : {metrics['wins']} / {metrics['losses']}")
        print(f"  Win Rate             : {metrics['win_rate']*100:.1f}%")
        print(f"  Total R              : {metrics['total_r']:+.2f} R")
        print(f"  Avg R per Trade      : {metrics['avg_r']:+.3f} R")
        print(f"  Profit Factor        : {metrics['profit_factor']:.2f}")
        print(f"  Max Drawdown         : {metrics['max_drawdown_r']:.2f} R")
        print(f"  Max Consecutive Loss : {metrics['max_consec_losses']}")
        print(f"  Avg Bars Held        : {metrics['avg_bars_held']:.0f} min")
        print()
        print("  By Scenario:")
        for sc, s in metrics["scenario_stats"].items():
            print(f"    {sc}: Count={s['n']}  WR={s['win_rate']*100:.0f}%  AvgR={s['avg_r']:+.3f}")
    print(sep)


  PERFORMANCE — NY Session M1 Breakout
  Total Trades         : 22
  Open (not closed)    : 0
  Wins / Losses        : 5 / 17
  Win Rate             : 22.7%
  Total R              : -4.50 R
  Avg R per Trade      : -0.205 R
  Profit Factor        : 0.73
  Max Drawdown         : -10.50 R
  Max Consecutive Loss : 7
  Avg Bars Held        : 166 min

  By Scenario:
    STRONG_BREAKOUT: Count=12  WR=25%  AvgR=-0.125
    FAKE_BREAK: Count=10  WR=20%  AvgR=-0.300


## Step 11 — Visualizations
### 11.1 — Equity Curve & Drawdown


In [236]:
def plot_equity(metrics: dict) -> None:
    if not metrics or "cum_r" not in metrics:
        print("No metrics to plot.")
        return
    cum_r  = metrics["cum_r"]
    dd     = metrics["drawdown"]
    closed = metrics["closed"].reset_index(drop=True)
    fig = make_subplots(
        rows=3, cols=1, row_heights=[0.5, 0.25, 0.25],
        subplot_titles=["Cumulative Equity (R)", "Drawdown (R)", "Per-Trade P&L (R)"],
        vertical_spacing=0.08,
    )
    fig.add_trace(go.Scatter(
        x=cum_r.index, y=cum_r.values, mode="lines", name="Equity (R)",
        line=dict(color="#00E5FF", width=2.5),
        fill="tozeroy", fillcolor="rgba(0,229,255,0.08)",
    ), row=1, col=1)
    fig.add_hline(y=0, line_color="gray", line_dash="dash", line_width=1, row=1, col=1)
    fig.add_trace(go.Scatter(
        x=dd.index, y=dd.values, mode="lines", name="Drawdown",
        line=dict(color="#FF1744", width=1.5),
        fill="tozeroy", fillcolor="rgba(255,23,68,0.15)",
    ), row=2, col=1)
    colors = ["#00E676" if r=="TP" else "#FF1744" for r in closed["result"]]
    fig.add_trace(go.Bar(x=closed.index, y=closed["pnl_r"],
                         marker_color=colors, name="Trade P&L"), row=3, col=1)
    fig.add_hline(y=0, line_color="gray", line_dash="dash", line_width=1, row=3, col=1)
    fig.update_layout(
        title=dict(
            text=(
                f"NY Session M1 Breakout — Equity Dashboard<br>"
                f"<sup>WR={metrics['win_rate']*100:.0f}%  "
                f"Total={metrics['total_r']:+.1f}R  "
                f"PF={metrics['profit_factor']:.2f}  "
                f"MaxDD={metrics['max_drawdown_r']:.1f}R</sup>"
            ),
            x=0.5,
        ),
        height=700, template="plotly_dark", showlegend=True,
    )
    fig.show()


plot_equity(metrics)


### 11.2 — Breakdown by Direction & Scenario


In [237]:
def plot_breakdown(trades_df: pd.DataFrame) -> None:
    closed = trades_df[trades_df["result"].isin(["TP","SL"])].copy()
    if closed.empty:
        return
    fig = make_subplots(rows=1, cols=3,
        subplot_titles=["Win Rate by Direction", "Win Rate by Scenario", "R Distribution"])
    for col_idx, grp_col, palette in [
        (1, "direction", {"BUY":"#26a69a", "SELL":"#ef5350"}),
        (2, "scenario",  {"STRONG_BREAKOUT":"#3d85c8", "FAKE_BREAK":"#e69138"}),
    ]:
        stats = closed.groupby(grp_col).apply(
            lambda x: pd.Series({"wr":(x["result"]=="TP").mean()*100,"n":len(x)})
        ).reset_index()
        colors = [palette.get(v,"#888") for v in stats[grp_col]]
        fig.add_trace(go.Bar(
            x=stats[grp_col], y=stats["wr"],
            text=[f"{r:.0f}%\n(n={n})" for r,n in zip(stats["wr"],stats["n"])],
            textposition="auto", marker_color=colors, name=grp_col,
        ), row=1, col=col_idx)
    fig.add_trace(go.Histogram(x=closed["pnl_r"], nbinsx=25,
                               marker_color="#7C4DFF", name="R Dist"), row=1, col=3)
    fig.add_vline(x=0, line_color="white", line_dash="dash", row=1, col=3)
    fig.update_layout(title="Performance Breakdown", height=400,
                      template="plotly_dark", showlegend=False)
    fig.show()


plot_breakdown(trades_df)


### 11.3 — Weekday Analysis


In [238]:
def plot_weekday_analysis(trades_df: pd.DataFrame) -> None:
    closed = trades_df[trades_df["result"].isin(["TP","SL"])].copy()
    if closed.empty:
        return
    closed["weekday"]  = pd.to_datetime(closed["date"]).dt.weekday
    days_map = {0:"Mon",1:"Tue",2:"Wed",3:"Thu",4:"Fri"}
    closed["day_name"] = closed["weekday"].map(days_map)
    stats = closed.groupby("day_name").agg(
        win_rate=("result", lambda x: (x=="TP").mean()*100),
        count=("result", "count"),
        avg_r=("pnl_r", "mean"),
    ).reindex(["Mon","Tue","Wed","Thu","Fri"])
    fig = make_subplots(rows=1, cols=2,
                        subplot_titles=["Win Rate by Day", "Avg R by Day"])
    fig.add_trace(go.Bar(
        x=stats.index, y=stats["win_rate"],
        text=[f"{v:.0f}%\n({n}tr)" for v,n in zip(stats["win_rate"],stats["count"])],
        textposition="auto", marker_color="#00BCD4", name="Win Rate",
    ), row=1, col=1)
    colors_r = ["#26a69a" if v>=0 else "#ef5350" for v in stats["avg_r"]]
    fig.add_trace(go.Bar(
        x=stats.index, y=stats["avg_r"],
        text=[f"{v:+.3f}R" for v in stats["avg_r"]],
        textposition="auto", marker_color=colors_r, name="Avg R",
    ), row=1, col=2)
    fig.add_hline(y=50, line_dash="dash", line_color="gray", row=1, col=1)
    fig.add_hline(y=0,  line_dash="dash", line_color="gray", row=1, col=2)
    fig.update_layout(title="Performance by Weekday", template="plotly_dark",
                      height=380, showlegend=False)
    fig.show()


plot_weekday_analysis(trades_df)


## Step 12 — Trade Review Dashboard
### Per-Trade Chart — M1

برای هر معامله: Zone (5 کندل اول 19:00)، Entry، SL، TP — همه روی **M1**.


In [239]:
def plot_trade(trade: pd.Series, df_m1: pd.DataFrame,
               pad_pre: int = 30, pad_post: int = 30) -> Optional[go.Figure]:
    """Full M1 trade chart: first-5-candle zone, entry/exit, SL, TP."""
    day    = trade["date"]
    day_m1 = df_m1[df_m1["date"] == day].reset_index(drop=True)

    entry_time = pd.Timestamp(trade["entry_time"])
    ei_cands   = day_m1[day_m1["time"] <= entry_time].index
    if len(ei_cands) == 0:
        return None
    ei = ei_cands[-1]

    exit_time = pd.Timestamp(trade["exit_time"]) if pd.notna(trade.get("exit_time")) else None
    if exit_time:
        xi_cands = day_m1[day_m1["time"] <= exit_time].index
        xi = xi_cands[-1] if len(xi_cands)>0 else len(day_m1)-1
    else:
        xi = len(day_m1)-1

    # Start view from 19:00 (zone open)
    zone_idx = day_m1[(day_m1["hour"]==NY_OPEN_HOUR) & (day_m1["minute"]==NY_OPEN_MIN)].index
    s_zone   = zone_idx[0] if len(zone_idx)>0 else max(0, ei-pad_pre)
    s = max(0, min(s_zone, ei-pad_pre))
    e = min(len(day_m1)-1, xi+pad_post)
    view = day_m1.iloc[s:e+1]

    fig = go.Figure()
    fig.add_trace(go.Candlestick(
        x=view["time"], open=view["open"], high=view["high"],
        low=view["low"], close=view["close"], name="M1",
        increasing_line_color="#26a69a", decreasing_line_color="#ef5350",
        increasing_fillcolor="#26a69a", decreasing_fillcolor="#ef5350",
    ))

    zh, zl = trade["zone_high"], trade["zone_low"]

    # Highlight zone candles
    zone_start = day_m1[(day_m1["hour"]==NY_OPEN_HOUR) & (day_m1["minute"]==NY_OPEN_MIN)]["time"]
    if not zone_start.empty:
        zone_end = zone_start.iloc[0] + pd.Timedelta(minutes=ZONE_CANDLES)
        fig.add_vrect(x0=zone_start.iloc[0], x1=zone_end,
                      fillcolor="rgba(33,150,243,0.20)", line_width=0,
                      annotation_text=f"Zone ({ZONE_CANDLES} M1)",
                      annotation_position="top left")

    fig.add_hrect(y0=zl, y1=zh, fillcolor="rgba(33,150,243,0.07)", line_width=0)
    fig.add_hline(y=zh, line_color="rgba(33,150,243,0.9)", line_dash="dash", line_width=2,
                  annotation_text=f"ZH {zh:.2f}", annotation_position="top right")
    fig.add_hline(y=zl, line_color="rgba(255,152,0,0.9)",  line_dash="dash", line_width=2,
                  annotation_text=f"ZL {zl:.2f}", annotation_position="bottom right")
    fig.add_hline(y=trade["sl"], line_color="rgba(255,23,68,0.85)", line_dash="dot", line_width=1.5,
                  annotation_text=f"SL {trade['sl']:.2f}", annotation_position="bottom left")
    fig.add_hline(y=trade["tp"], line_color="rgba(0,230,118,0.85)",  line_dash="dot", line_width=1.5,
                  annotation_text=f"TP {trade['tp']:.2f}", annotation_position="top left")

    ec   = "#26a69a" if trade["direction"]=="BUY" else "#ef5350"
    esym = "triangle-up"   if trade["direction"]=="BUY" else "triangle-down"
    etp  = "top center"    if trade["direction"]=="BUY" else "bottom center"
    fig.add_trace(go.Scatter(
        x=[trade["entry_time"]], y=[trade["entry_price"]],
        mode="markers+text", name="Entry",
        marker=dict(symbol=esym, size=20, color=ec, line=dict(color="white",width=2)),
        text=[f"ENTRY\n{trade['entry_price']:.2f}"],
        textposition=etp, textfont=dict(size=10, color="white"),
    ))

    result = trade.get("result")
    if result in ("TP","SL") and exit_time:
        xc   = "#00E676" if result=="TP" else "#FF1744"
        xsym = "star"    if result=="TP" else "x"
        fig.add_trace(go.Scatter(
            x=[trade["exit_time"]], y=[trade["exit_price"]],
            mode="markers+text", name=f"Exit ({result})",
            marker=dict(symbol=xsym, size=18, color=xc, line=dict(color="white",width=2)),
            text=[f"{result}\n{trade['exit_price']:.2f}"],
            textposition="top center", textfont=dict(size=10, color="white"),
        ))

    emoji = {"TP":"✅","SL":"❌","OPEN":"⏳"}.get(result,"?")
    pnl   = trade.get("pnl_r", 0)
    fig.update_layout(
        title=(
            f"{emoji} {trade['direction']} — {trade['scenario']} — {day}  |  "
            f"PnL: {pnl:+.2f}R  |  SL={trade['sl']:.2f}  TP={trade['tp']:.2f}"
        ),
        xaxis_rangeslider_visible=False,
        template="plotly_dark", height=520,
        yaxis_title="Price (USD)", xaxis_title="Time (broker CSV)",
    )
    return fig


print("plot_trade() defined.")


plot_trade() defined.


In [240]:
N_SHOW = min(6, len(trades_df))

for i in range(N_SHOW):
    t   = trades_df.iloc[i]
    fig = plot_trade(t, df_m1)
    if fig:
        fig.show()
    status = t.get("result","?")
    pnl    = t.get("pnl_r", 0)
    print(f"  Trade {i+1:2d}: {t['direction']:4s} {t['scenario']:22s}  "
          f"{t['date']}  →  {status:4s}  {pnl:+.2f}R")


  Trade  1: BUY  STRONG_BREAKOUT         2026-04-16  →  SL    -1.00R


  Trade  2: BUY  STRONG_BREAKOUT         2026-04-17  →  SL    -1.00R


  Trade  3: SELL STRONG_BREAKOUT         2026-04-20  →  SL    -1.00R


  Trade  4: BUY  STRONG_BREAKOUT         2026-04-21  →  SL    -1.00R


  Trade  5: SELL STRONG_BREAKOUT         2026-04-22  →  TP    +2.50R


  Trade  6: SELL FAKE_BREAK              2026-04-23  →  TP    +2.50R


### Overview Dashboard — All Trades on M1


In [241]:
def plot_overview(trades_df: pd.DataFrame, df_m1: pd.DataFrame) -> None:
    """All trade entry markers overlaid on M1 chart."""
    if trades_df.empty:
        print("No trades to display.")
        return
    fig = go.Figure()
    fig.add_trace(go.Candlestick(
        x=df_m1["time"], open=df_m1["open"], high=df_m1["high"],
        low=df_m1["low"], close=df_m1["close"],
        name="XAUUSD M1",
        increasing_line_color="#26a69a", decreasing_line_color="#ef5350",
    ))
    styles = {
        ("BUY",  "TP")  : dict(symbol="triangle-up",        size=14, color="#00E676", name="BUY TP"),
        ("BUY",  "SL")  : dict(symbol="triangle-up-open",   size=14, color="#FF9800", name="BUY SL"),
        ("SELL", "TP")  : dict(symbol="triangle-down",      size=14, color="#00B0FF", name="SELL TP"),
        ("SELL", "SL")  : dict(symbol="triangle-down-open", size=14, color="#FF1744", name="SELL SL"),
        ("BUY",  "OPEN"): dict(symbol="circle", size=10, color="white", name="OPEN"),
        ("SELL", "OPEN"): dict(symbol="circle", size=10, color="white", name="OPEN"),
    }
    seen = set()
    for (direction, result), style in styles.items():
        sub = trades_df[(trades_df["direction"]==direction) & (trades_df["result"]==result)]
        if sub.empty:
            continue
        key  = (direction, result)
        name = style.pop("name")
        fig.add_trace(go.Scatter(
            x=sub["entry_time"], y=sub["entry_price"],
            mode="markers", name=name,
            marker=dict(**style, line=dict(color="white",width=1)),
            showlegend=key not in seen,
        ))
        seen.add(key)
    fig.update_layout(
        title="All Trades — NY Session M1 Breakout — XAUUSD M1 Overview",
        xaxis_rangeslider_visible=False,
        template="plotly_dark", height=600,
        legend=dict(orientation="h", yanchor="bottom", y=1.01),
    )
    fig.show()


plot_overview(trades_df, df_m1)


### Zone Width Analysis


In [242]:
def plot_zone_width_analysis(trades_df: pd.DataFrame) -> None:
    closed = trades_df[trades_df["result"].isin(["TP","SL"])].copy()
    if closed.empty:
        return
    fig = make_subplots(rows=1, cols=2,
        subplot_titles=["Zone Width vs PnL (R)", "Zone Width by Result"])
    colors = ["#00E676" if r=="TP" else "#FF1744" for r in closed["result"]]
    fig.add_trace(go.Scatter(
        x=closed["zone_width"], y=closed["pnl_r"],
        mode="markers", marker=dict(color=colors, size=8, opacity=0.75),
        name="Trades",
    ), row=1, col=1)
    fig.add_hline(y=0, line_dash="dash", line_color="gray", row=1, col=1)
    for result, color in [("TP","#00E676"),("SL","#FF1744")]:
        sub = closed[closed["result"]==result]
        fig.add_trace(go.Box(y=sub["zone_width"], name=result, marker_color=color), row=1, col=2)
    fig.update_layout(
        title=f"Zone Width Analysis (first {ZONE_CANDLES} M1 candles)",
        template="plotly_dark", height=420,
        xaxis_title="Zone Width (USD)", yaxis_title="PnL (R)", showlegend=False,
    )
    fig.show()


plot_zone_width_analysis(trades_df)


### Trade Table


In [243]:
cols_show = ["date","direction","scenario","entry_price","sl","tp",
             "risk","zone_high","zone_low","zone_width",
             "result","exit_price","pnl_r","bars_held"]
avail = [c for c in cols_show if c in trades_df.columns]
print(f"All {len(trades_df)} trades:")
pd.set_option("display.max_rows", 200)
print(trades_df[avail].to_string(index=False))


All 22 trades:
      date direction        scenario  entry_price         sl         tp     risk  zone_high   zone_low  zone_width result  exit_price   pnl_r  bars_held
2026-04-16       BUY STRONG_BREAKOUT   74871.0500 74443.6000 75939.6800 427.4500 74801.4900 74683.1000    118.3900     SL  74443.6000 -1.0000         25
2026-04-17       BUY STRONG_BREAKOUT   78002.2000 77690.9500 78780.3200 311.2500 77908.1300 77691.2500    216.8800     SL  77690.9500 -1.0000         61
2026-04-20      SELL STRONG_BREAKOUT   75448.9500 75717.1700 74778.4000 268.2200 75710.6400 75503.4400    207.2000     SL  75717.1700 -1.0000         69
2026-04-21       BUY STRONG_BREAKOUT   76128.1200 75817.9400 76903.5700 310.1800 75987.9600 75818.2400    169.7200     SL  75817.9400 -1.0000         23
2026-04-22      SELL STRONG_BREAKOUT   79178.5700 79459.6400 78475.9000 281.0700 79422.9300 79259.8300    163.1000     TP  78475.9000  2.5000        251
2026-04-23      SELL      FAKE_BREAK   78295.3900 78497.2800 77790.

## Final Analysis


In [244]:
if trades_df.empty:
    print("No trades to analyze.")
elif metrics.get("total_trades", 0) == 0:
    print("No closed trades to analyze.")
    print(f"  Open trades : {metrics.get('open_trades', 0)}")
    print("  All entries were generated but SL/TP not hit within MAX_TRADE_HOURS.")
    print("  Possible causes:")
    print("    1. Zone is too wide → risk too large for price to move 2R quickly")
    print("    2. Check if symbol has data at 19:00 (run the session detection cell)")
    print("    3. Try increasing MAX_TRADE_HOURS or reducing RISK_REWARD")
else:
    closed = trades_df[trades_df["result"].isin(["TP","SL"])]
    print("=" * 60)
    print("FINAL ANALYSIS — NY Session M1 Breakout")
    print(f"Zone: first {ZONE_CANDLES} M1 candles at 19:00")
    print("=" * 60)
    wr = metrics.get("win_rate", 0)
    pf = metrics.get("profit_factor", 0)
    tr = metrics.get("total_r", 0)
    dd = metrics.get("max_drawdown_r", 0)
    breakeven_wr = 1 / (1 + RISK_REWARD)
    print(f"
[Expectancy]")
    print(f"  Break-even WR at 1:{RISK_REWARD} RR = {breakeven_wr*100:.1f}%")
    print(f"  Actual Win Rate       = {wr*100:.1f}%")
    edge = "POSITIVE EDGE" if wr > breakeven_wr else "NEGATIVE EDGE"
    print(f"  Assessment            = {edge}")
    sc_stats = metrics.get("scenario_stats", {})
    if sc_stats:
        print(f"
[Best Performing Scenario]")
        best = max(sc_stats.items(), key=lambda x: x[1]["avg_r"])
        print(f"  {best[0]}: WR={best[1]['win_rate']*100:.0f}%  AvgR={best[1]['avg_r']:+.3f}")
    print(f"
[Risk Profile]")
    print(f"  Max Drawdown     = {dd:.1f}R")
    print(f"  Profit Factor    = {pf:.2f}")
    print(f"  Total R          = {tr:+.2f}R over {LOOKBACK_DAYS} days")
    if metrics.get("total_trades", 0) > 0:
        tpw = metrics["total_trades"] / (LOOKBACK_DAYS / 7)
        print(f"  Trade Frequency  = {tpw:.1f} trades/week")


SyntaxError: unterminated f-string literal (detected at line 22) (3428836609.py, line 22)